In [1]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/50_Startups.csv")

print(df.shape)

print(df.columns)
df.head()
print(df.describe())

(50, 5)
Index(['R&D Spend', 'Administration', 'Marketing Spend', 'State', 'Profit'], dtype='object')
           R&D Spend  Administration  Marketing Spend         Profit
count      50.000000       50.000000        50.000000      50.000000
mean    73721.615600   121344.639600    211025.097800  112012.639200
std     45902.256482    28017.802755    122290.310726   40306.180338
min         0.000000    51283.140000         0.000000   14681.400000
25%     39936.370000   103730.875000    129300.132500   90138.902500
50%     73051.080000   122699.795000    212716.240000  107978.190000
75%    101602.800000   144842.180000    299469.085000  139765.977500
max    165349.200000   182645.560000    471784.100000  192261.830000


In [2]:
URL = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/asia_documents_csv.csv"
df = pd.read_csv(URL)
print(df.shape)
print(df.columns)
print(df.describe())
df.head()

(10, 3)
Index(['id', 'source', 'content'], dtype='object')
             id source                                            content
count        10     10                                                 10
unique       10      9                                                 10
top     Japan_1  Japan  Japan is an island nation in East Asia, known ...
freq          1      2                                                  1


,id,source,content
0,Japan_1,Japan,"Japan is an island nation in East Asia, known ..."
1,Japan_2,Japan,scrapers.
2,Taiwan_1,Taiwan,Taiwan is an island located off the southeaste...
3,Thailand_1,Thailand,Thailand is a Southeast Asian country famous f...
4,South_Korea_1,South_Korea,"South Korea is a dynamic country in East Asia,..."


In [6]:
# Importamos requests para hacer peticiones HTTP a la API de GitHub.
import requests

# BytesIO permite que pandas lea desde los bytes descargados en memoria.
from io import BytesIO

# NumPy se utiliza aquí para aplicar logaritmos en la puntuación de complejidad.
import numpy as np

# La API de GitHub permite consultar el contenido de un repositorio sin
# tener que conocer de antemano los nombres de todos sus archivos.
# El parámetro recursive=1 hace que también se revisen las carpetas internas.
api_url = "https://api.github.com/repos/gakudo-ai/open-datasets/git/trees/main?recursive=1"

# Enviamos la petición y guardamos la respuesta del servidor.
response = requests.get(api_url)

# Si GitHub devuelve un error HTTP, esta línea detiene la ejecución y muestra
# el problema en lugar de continuar trabajando con datos incompletos.
response.raise_for_status()

# Convertimos la respuesta JSON en una lista de rutas de archivos.
# Nos quedamos únicamente con los elementos que:
#   1. Son archivos (type == blob).
#   2. Terminan en .csv, sin importar si la extensión usa mayúsculas.
csv_paths = [
    item["path"]
    for item in response.json()["tree"]
    if item["type"] == "blob" and item["path"].lower().endswith(".csv")
]

# Este diccionario conservará cada DataFrame usando la ruta del archivo como
# clave. Así podremos reutilizar posteriormente los datos ya descargados.
csv_frames = {}

# Esta lista almacenará un resumen de las métricas calculadas para cada CSV.
# Después se convertirá en un DataFrame comparativo.
results = []

# Recorremos todos los archivos CSV localizados en el repositorio.
for path in csv_paths:
    # Construimos la URL directa del archivo sin descargarlo manualmente.
    # La ruta debe conservar las subcarpetas que pueda tener el repositorio.
    csv_url = f"https://raw.githubusercontent.com/gakudo-ai/open-datasets/main/{path}"
    
    try:
        # Descargamos el archivo como contenido binario para poder medir su
        # tamaño exacto antes de transformarlo en un DataFrame.
        csv_response = requests.get(csv_url)
        csv_response.raise_for_status()

        # len devuelve el número de bytes descargados.
        # Dividimos entre 1024**2 para expresarlo en megabytes (MB).
        file_size_mb = len(csv_response.content) / (1024 ** 2)

        # Leemos el CSV desde los bytes descargados en memoria.
        # low_memory=False ayuda a que pandas infiera los tipos de forma más
        # consistente en archivos grandes o con columnas heterogéneas.
        # encoding_errors="replace" sustituye caracteres problemáticos para
        # evitar que un error de codificación interrumpa todo el recorrido.
        data = pd.read_csv(
            BytesIO(csv_response.content),
            low_memory=False,
            encoding_errors="replace"
        )

        # Guardamos el DataFrame completo para poder consultarlo después.
        csv_frames[path] = data

        # Seleccionamos las columnas cuyo tipo es object, que normalmente
        # representan texto en pandas.
        text_columns = data.select_dtypes(include="object").columns

        # Calculamos la longitud media de los valores de texto.
        # astype(str) permite medir también valores que no sean cadenas.
        # apply calcula la longitud de cada valor y mean obtiene la media
        # de cada columna; el segundo mean combina esas medias.
        # Si no hay columnas de texto, usamos 0 para evitar un resultado vacío.
        average_text_length = (
            data[text_columns].astype(str).apply(lambda col: col.str.len()).mean().mean()
            if len(text_columns) > 0 else 0
        )

        # isna identifica los valores ausentes de toda la tabla.
        # La primera media calcula el porcentaje de ausencias por columna y
        # la segunda media obtiene el porcentaje medio de toda la tabla.
        missing_ratio = data.isna().mean().mean()

        # Contamos cuántos tipos de datos distintos aparecen en el DataFrame.
        # Una mayor variedad de tipos suele requerir más trabajo de análisis.
        dtype_count = data.dtypes.nunique()

        # Construimos una puntuación orientativa de complejidad.
        # - log1p reduce el peso de los datasets extremadamente grandes.
        # - El tamaño depende de filas y columnas.
        # - dtype_count añade complejidad por variedad de tipos.
        # - La longitud del texto añade complejidad por contenido textual.
        # - Los valores ausentes reciben un peso de 5 porque suelen exigir
        #   limpieza o decisiones adicionales durante el análisis.
        complexity_score = (
            np.log1p(len(data)) *
            np.log1p(len(data.columns)) +
            2 * dtype_count +
            np.log1p(average_text_length) +
            5 * missing_ratio
        )

        # Guardamos una fila de resumen con las métricas principales del CSV.
        # El tamaño del archivo se añade como columna independiente para que
        # pueda compararse sin modificar la puntuación de complejidad.
        # round hace que los valores decimales sean más fáciles de leer.
        results.append({
            "archivo": path,
            "tamaño_MB": round(file_size_mb, 2),
            "filas": len(data),
            "columnas": len(data.columns),
            "tipos_de_datos": dtype_count,
            "valores_ausentes_%": round(missing_ratio * 100, 2),
            "longitud_texto_promedio": round(average_text_length, 2),
            "puntuacion_complejidad": round(complexity_score, 2)
        })

    except Exception as error:
        # Si un archivo concreto no puede descargarse o leerse, mostramos el
        # error y continuamos con los demás archivos del repositorio.
        print(f"No se pudo leer {path}: {error}")

# Convertimos la lista de diccionarios en un DataFrame.
# sort_values ordena de menor a mayor complejidad y reset_index crea
# un índice consecutivo después de ordenar.
comparacion = (
    pd.DataFrame(results)
    .sort_values("puntuacion_complejidad")
    .reset_index(drop=True)
)

# Mostramos la tabla completa para comparar todos los datasets.
display(comparacion)

# La primera fila corresponde al dataset con menor puntuación.
dataset_mas_simple = comparacion.iloc[0]

# La última fila corresponde al dataset con mayor puntuación.
dataset_mas_complejo = comparacion.iloc[-1]

# Presentamos por separado el dataset más sencillo de analizar.
print("Dataset más simple:")
display(dataset_mas_simple.to_frame().T)

# Presentamos por separado el dataset con mayor complejidad estimada.
print("Dataset más complejo:")
display(dataset_mas_complejo.to_frame().T)

HTTPError: 403 Client Error: rate limit exceeded for url: https://api.github.com/repos/gakudo-ai/open-datasets/git/trees/main?recursive=1